# 3 · Validate the agent (smoke test against the real, persistent index)

Requires notebook 1 to have been run first — this calls `portfolio.load_all_indexes()`,
the same fast, no-rebuild load `mcp_server.py`/`web_app.py` use, and it will raise a
clear error if the index hasn't been provisioned yet. That's deliberate: this notebook
is meant to test the actual thing the live services depend on, not a separate copy of
it.

Runs every question in `data/eval_set_starter.csv` through the real agent (the same
`build_agent()` both live services call) and checks two things: did it route to the
right project(s), and does the answer actually mention the reference fact — not just
"did it return something," a real correctness check.

In [ ]:
%%capture
!pip install -q -r requirements.txt


In [ ]:
# Get the project files (config.py, portfolio.py, data/) if they aren't
# already here -- lets this notebook be opened and run on its own in Colab.
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


In [ ]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")

# QDRANT_URL/QDRANT_API_KEY make the index PERSISTENT (Qdrant Cloud, free
# tier is enough) instead of rebuilt from scratch every time. This matters
# specifically because this notebook runs in a fresh Colab VM each time,
# separate from wherever mcp_server.py/web_app.py actually run - without a
# real, shared QDRANT_URL, this notebook's work never reaches those
# services at all. Get a free instance at https://cloud.qdrant.io
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL (blank = local in-memory, no persistence): ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


In [ ]:
import config
import portfolio

indexes = portfolio.load_all_indexes()  # requires notebook 1 to have been run - raises a clear error otherwise
router = portfolio.build_router()
agent = portfolio.build_agent(indexes, router)
print("Agent ready, using the persisted index.")


In [ ]:
import pandas as pd

eval_questions = pd.read_csv("data/eval_set_starter.csv")
print(f"{len(eval_questions)} regression questions loaded")
eval_questions[["question", "expected_projects"]]


Run every regression question through the agent, and use Gemini itself as a judge
against the `reference_fact` column — matching the LLM-judge pattern already used
elsewhere in this portfolio (allam_finetune's evaluation), not a new one invented here.

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
JUDGE_MODEL = "gemini-3.1-flash-lite"


def judge_answer_covers_fact(answer, reference_fact):
    prompt = (
        f"Does this answer correctly convey the following fact? Reply with ONLY "
        f"YES or NO.\n\nFact: {reference_fact}\n\nAnswer: {answer}"
    )
    response = client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    return response.text.strip().upper().startswith("YES")


results = []
for _, row in eval_questions.iterrows():
    expected = [p.strip() for p in row["expected_projects"].split(",")]
    result = portfolio.ask(agent, row["question"])
    routed_correctly = all(p in result["projects"] for p in expected)
    fact_covered = judge_answer_covers_fact(result["answer"], row["reference_fact"])

    results.append({
        "question": row["question"][:60],
        "expected": expected,
        "routed_correctly": routed_correctly,
        "fact_covered": fact_covered,
        "retries": result["retries"],
    })

results_df = pd.DataFrame(results)
print(f"Routed correctly: {results_df['routed_correctly'].sum()}/{len(results_df)}")
print(f"Fact covered:     {results_df['fact_covered'].sum()}/{len(results_df)}")
results_df


**A failure here means the live services would give a wrong or ungrounded answer
to a real question** — this is the actual correctness gate, not a demo. `retries > 0` on
a passing row is fine (that's the critique loop doing its job); a failing row after
retries are exhausted is the one worth looking at closely.